# Notebook 12.3 — PPO for Reasoning (PyTorch and `trl`), and the Final Comparison

### What this notebook does

1. **PPO from scratch in PyTorch** — the InstructGPT-style algorithm from
   notebooks 6–7, on the same reasoning task and the **same verifier reward** as
   notebook 12's GRPO. A value head (critic), per-token rewards, GAE, clipped
   policy and value losses, several epochs per batch.
2. **PPO with `trl`** — `trl.experimental.ppo.PPOTrainer`, with the verifier
   wrapped as an `nn.Module` "reward model".
3. **The final comparison** of every method across notebooks 12, 12.2 and 12.3:
   accuracy, format, length, which problems each fixed, cost, and what each
   algorithm actually needs.

Because notebook 12 and this notebook use the **same reward** (format +
correctness), **same model**, **same LoRA**, **same sample budget** and the
**same test set**, *GRPO vs PPO* is a clean comparison of the algorithms
themselves.

### PPO vs GRPO in one table

| | GRPO (notebook 12) | PPO (this notebook) |
|---|---|---|
| samples per prompt | a **group** of $G = 8$ | **one** |
| baseline / credit assignment | group mean reward; every token of an answer shares one advantage | a learned **critic** $V(s_t)$; **GAE** gives each token its own advantage |
| extra model | none | a value head (here) or a separate value model (`trl`) |
| KL to the reference | k3, added to the **loss** | k1, subtracted from the **reward** at every token |
| reuse of each batch | 1 step (`MU = 1`) | several epochs × minibatches → the clip matters |

> **`trl` version.** `PPOTrainer` lives in `trl.experimental.ppo` and was
> **removed after `trl` 1.12**. The `trl` part needs `pip install "trl==1.12.0"`,
> which works with recent `transformers`. If the import fails, that part is
> skipped automatically. The hand-written PPO and the comparison still run.

> **Hardware.** Written for a CUDA GPU (bf16 when supported). Roughly an hour
> on an A100. Reduce `PPO_STEPS` on smaller GPUs.

## 1. Setup

In [ ]:
# ---- Environment setup (uncomment what you need, then Runtime -> Restart session) ----
!pip install -q "transformers>=4.56" "peft>=0.17" datasets accelerate "trl==1.12.0"   # PPOTrainer was removed after 1.12
#
# Colab-style images ship an old `torchao`; recent `peft` refuses to create LoRA layers when an incompatible
# torchao (< 0.16) is installed. Either remove it or upgrade it:
!pip uninstall -y torchao

In [ ]:
import os, re, time, random, copy, json, hashlib, gc, warnings
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

import transformers, trl, peft
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification, AutoModel
from peft import LoraConfig, get_peft_model

try:
    from trl.experimental.ppo import PPOConfig, PPOTrainer
    HAS_TRL_PPO = True
except ImportError:
    HAS_TRL_PPO = False
    print('NOTE: trl.experimental.ppo not found -> the trl PPO part will be skipped. Install with: pip install "trl==1.12.0"')


# ---- pick the fastest available device: CUDA GPU > Apple-Silicon GPU > CPU ----
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
USE_BF16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported()      # bf16: half the memory, much faster on modern GPUs
DTYPE = torch.bfloat16 if USE_BF16 else torch.float32

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
set_seed(0)

def free_memory():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

print(f"device        : {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""))
print(f"dtype         : {DTYPE}")
print(f"torch         : {torch.__version__}")
print(f"transformers  : {transformers.__version__}")
print(f"trl           : {trl.__version__}")
print(f"peft          : {peft.__version__}")

## 2. Task, verifier, evaluation and reward — identical to notebook 12

In [ ]:
# =====================  identical to notebook 12  =====================
NAMES = ["Sam", "Mia", "Leo", "Ava", "Noah", "Zoe", "Eli", "Ivy"]
ITEMS = ["apples", "marbles", "stickers", "books", "cookies", "pencils"]

def make_problem(rng):
    """Return (question, answer_as_string). Three templates, each needing 2-3 arithmetic steps."""
    name, item = rng.choice(NAMES), rng.choice(ITEMS)
    a, b, c, d = rng.randint(2, 20), rng.randint(2, 6), rng.randint(2, 9), rng.randint(1, 10)
    kind = rng.randint(0, 2)
    if kind == 0:
        q = (f"{name} has {a} {item}. {name} buys {b} packs with {c} {item} in each pack, "
             f"then gives away {d}. How many {item} does {name} have now?")
        ans = a + b * c - d
    elif kind == 1:
        q = (f"A box holds {c} {item}. {name} fills {b} boxes and has {a} loose {item}. "
             f"{name} then loses {d}. How many {item} are left?")
        ans = b * c + a - d
    else:
        q = (f"{name} reads {c} pages a day for {b} days, then reads {a} more pages. "
             f"How many pages did {name} read in total?")
        ans = b * c + a
    return q, str(ans)

def make_dataset(n, seed):
    rng = random.Random(seed)
    return [make_problem(rng) for _ in range(n)]

TRAIN = make_dataset(400, seed=1)
train_questions = {q for q, _ in TRAIN}
TEST = [p for p in make_dataset(60, seed=123) if p[0] not in train_questions][:48]
ANSWER_OF = {q: a for q, a in TRAIN + TEST}

# A fingerprint of the test set: notebook 12.3 refuses to compare results computed on different test sets.
TEST_FINGERPRINT = hashlib.sha1("\n".join(f"{q}|{a}" for q, a in TEST).encode()).hexdigest()[:12]
print(f"train problems: {len(TRAIN)} | test problems: {len(TEST)} | overlap: {len(train_questions & {q for q, _ in TEST})}"
      f" | test fingerprint: {TEST_FINGERPRINT}")

SYSTEM_PROMPT = (
    "You are a helpful assistant. First think through the problem step by step inside "
    "<think> </think> tags, then give only the final number inside <answer> </answer> tags."
)
ANSWER_RE = re.compile(r"<answer>\s*(-?\d+)\s*</answer>")
FORMAT_RE = re.compile(r"<think>.+?</think>\s*<answer>\s*-?\d+\s*</answer>", re.S)

def parse_answer(text):
    """Strict verifier: the number inside <answer>...</answer>, else None."""
    m = ANSWER_RE.search(text)
    return m.group(1) if m else None

def follows_format(text):
    return bool(FORMAT_RE.search(text))

def format_score(text):
    r = 0.0
    if re.search(r"<think>.+?</think>", text, re.S):
        r += 0.10                                   # a reasoning block (worth little on its own)
    if ANSWER_RE.search(text):
        r += 0.25                                   # commits to a parseable answer
    if follows_format(text) and text.strip().endswith("</answer>"):
        r += 0.40                                   # the full structure: think -> answer -> stop
    return r

def correctness_score(text, answer):
    return 1.0 if parse_answer(text) == answer else 0.0

def verifier_reward(text, answer):
    """The notebook-12 reward: format (max 0.75) + correctness (1.0)."""
    return format_score(text) + correctness_score(text, answer)

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"          # left-pad prompts so generation continues on the right
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def chat_prompt(question, system=SYSTEM_PROMPT):
    """System prompt + user question, rendered with the model's own chat template (ready for generation)."""
    messages = [{"role": "system", "content": system}, {"role": "user", "content": question}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def build_prompt(question):
    return chat_prompt(question, SYSTEM_PROMPT)

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE).to(DEVICE).eval()
print(f"loaded {MODEL_NAME}: {sum(p.numel() for p in base_model.parameters()) / 1e6:.0f}M parameters on {DEVICE}")

@torch.no_grad()
def generate_texts(model, questions, system=SYSTEM_PROMPT, sample=True, temperature=0.8,
                   max_new_tokens=200, batch_size=32, seed=None):
    """Generate one completion per question (sampled or greedy)."""
    if seed is not None:
        torch.manual_seed(seed)
    model.eval()
    outs = []
    for i in range(0, len(questions), batch_size):
        enc = tokenizer([chat_prompt(q, system) for q in questions[i:i + batch_size]],
                        return_tensors="pt", padding=True).to(model.device)
        kw = dict(do_sample=True, temperature=temperature, top_k=0, top_p=1.0) if sample else dict(do_sample=False)
        out = model.generate(**enc, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id, **kw)
        outs += tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs

@torch.no_grad()
def evaluate(model, label, problems=TEST, max_new_tokens=200, batch_size=16, show=2):
    """Identical protocol to notebook 12: held-out problems, same system prompt, GREEDY decoding."""
    t0 = time.time()
    responses = generate_texts(model, [q for q, _ in problems], sample=False,
                               max_new_tokens=max_new_tokens, batch_size=batch_size)
    answers = [a for _, a in problems]
    metrics = {
        "accuracy (strict)": float(np.mean([parse_answer(r) == a for r, a in zip(responses, answers)])),
        "format compliance": float(np.mean([follows_format(r) for r in responses])),
        "accuracy (lenient, last number)": float(np.mean([re.findall(r"-?\d+", r)[-1:] == [a] for r, a in zip(responses, answers)])),
        "mean length (tokens)": float(np.mean([len(tokenizer(r)["input_ids"]) for r in responses])),
    }
    print(f"===== {label}: evaluated {len(problems)} held-out problems in {time.time() - t0:.0f}s =====")
    for k, v in metrics.items():
        print(f"   {k:34}: {v:.3f}" if "length" not in k else f"   {k:34}: {v:.1f}")
    for j in range(show):
        print(f"\n   --- example {j + 1}: {problems[j][0]}\n   --- true answer: {problems[j][1]}\n{responses[j]}")
    return metrics, responses

def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)
CKPT_DIR = Path("checkpoints"); CKPT_DIR.mkdir(exist_ok=True)

def method_record(metrics, responses, *, algorithm, implementation, reward_signal, train_minutes,
                  trainable_params, samples_generated, models_in_memory, notes=""):
    return {"metrics": {k: float(v) for k, v in metrics.items()}, "responses": list(responses),
            "algorithm": algorithm, "implementation": implementation, "reward_signal": reward_signal,
            "train_minutes": None if train_minutes is None else float(train_minutes),
            "trainable_params": None if trainable_params is None else int(trainable_params),
            "samples_generated": None if samples_generated is None else int(samples_generated),
            "models_in_memory": models_in_memory, "notes": notes}

def save_results(filename, methods):
    path = RESULTS_DIR / filename
    payload = {"notebook": filename, "model": MODEL_NAME, "test_fingerprint": TEST_FINGERPRINT,
               "test": [list(p) for p in TEST], "methods": methods}
    path.write_text(json.dumps(payload, indent=1))
    print(f"saved {len(methods)} method(s) to {path.resolve()}")

def selective_log_softmax(logits, targets):
    """log p(target) = z[target] - logsumexp(z), row by row: never builds a (B, T, 151936) log-prob tensor."""
    return torch.stack([row.gather(-1, tgt[:, None]).squeeze(-1).float() - torch.logsumexp(row.float(), dim=-1)
                        for row, tgt in zip(logits, targets)])

def lora_cfg(task_type="CAUSAL_LM"):
    return LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, target_modules="all-linear", task_type=task_type)

def lora_params_to_fp32(model):
    """Keep the (small) trainable LoRA weights in float32: Adam updates on bf16 weights lose precision."""
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()


before_metrics, before_responses = evaluate(base_model, "BEFORE training (base model)")

# Part 1 — PPO from scratch: `nn.Module` + PyTorch training loop

### The model: policy and critic sharing one backbone

`PolicyValueModel` wraps the LoRA-adapted language model and adds a **value
head**: one linear layer reading the last hidden state at every position.

- **Policy** $\pi_\theta(o_t \mid s_t)$: the LM's next-token distribution, where state
  $s_t$ = prompt + the first $t$ tokens of the answer.
- **Critic** $V_\psi(s_t)$: "how much reward do I expect from here?" It's computed
  from the hidden state *before* token $t$ is generated, i.e. position
  `n_prompt - 1 + t`, the same position whose logits predict token $t$.
- **Reference** $\pi_{\text{ref}}$: the same network with LoRA disabled (notebook 12, FAQ 3).

Sharing the backbone saves memory, which is what the old `trl`
`AutoModelForCausalLMWithValueHead` did. InstructGPT, and `trl`'s current PPO,
use a separate value network instead (Part 2).

### The rewards: sparse score + per-token KL penalty (notebook 7)

$$
r_t = -\beta \left( \log \pi_{\text{old}}(o_t \mid s_t) - \log \pi_{\text{ref}}(o_t \mid s_t) \right) + \mathbb{1}[t = \text{last}] \cdot R(\text{answer}),
\qquad \beta = 0.04 ,
$$

where $R$ is notebook 12's verifier reward (format + correctness). The KL term is
the **k1** estimate of the reverse KL, $\mathrm{KL}(\pi_\theta \Vert \pi_{\text{ref}})$, spread over tokens.

### Advantages: GAE over the tokens of the answer (notebook 5)

$$
\delta_t = r_t + V(s_{t+1}) - V(s_t), \qquad \hat{A}_t = \delta_t + \lambda \hat{A}_{t+1}, \qquad \lambda = 0.95, \ \gamma = 1,
$$

with $V(s_{\text{after last}}) = 0$. The critic's regression target is the return $\hat{A}_t + V(s_t)$.

### The loss (notebook 6)

$$
\mathcal{L} = \underbrace{\mathbb{E}_t \left[ \max\left( -\hat{A}_t \rho_t , \ -\hat{A}_t \operatorname{clip}(\rho_t, 0.8, 1.2) \right) \right]}_{\text{clipped policy loss}}
+ c_v \, \underbrace{\mathbb{E}_t \left[ \tfrac12 \max\left( (V - R_t)^2 , \ (V^{\text{clip}} - R_t)^2 \right) \right]}_{\text{clipped value loss}},
\qquad c_v = 0.1 .
$$

Unlike notebook 12 (`MU = 1`), PPO reuses each batch for **2 epochs × 4
minibatches = 8 updates**. So $\rho_t$ drifts away from 1, and the clip does real work.

In [ ]:
class PolicyValueModel(nn.Module):
    """LoRA policy + linear value head on the shared backbone. Reference = adapters disabled."""

    def __init__(self, model_name):
        super().__init__()
        base = AutoModelForCausalLM.from_pretrained(model_name, dtype=DTYPE)
        self.lm = get_peft_model(base, lora_cfg())
        lora_params_to_fp32(self.lm)
        self.value_head = nn.Linear(base.config.hidden_size, 1)                 # float32
        nn.init.zeros_(self.value_head.weight); nn.init.zeros_(self.value_head.bias)

    @torch.no_grad()
    def sample(self, questions, max_new_tokens=160, temperature=1.0):
        """Rollout: ONE sampled completion per question (PPO needs no groups)."""
        enc = tokenizer([build_prompt(q) for q in questions], return_tensors="pt", padding=True).to(DEVICE)
        seqs = self.lm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature,
                                top_k=0, top_p=1.0, pad_token_id=tokenizer.pad_token_id)
        n_prompt = enc["input_ids"].shape[1]
        comp = seqs[:, n_prompt:]
        is_eos = comp == tokenizer.eos_token_id
        last = torch.where(is_eos.any(1), is_eos.float().argmax(1), torch.full((len(comp),), comp.shape[1] - 1, device=DEVICE))
        cmask = (torch.arange(comp.shape[1], device=DEVICE)[None] <= last[:, None]).float()
        attn = torch.cat([enc["attention_mask"], cmask.long()], dim=1)
        texts = tokenizer.batch_decode(comp, skip_special_tokens=True)
        return seqs, attn, cmask, n_prompt, last, texts

    def forward(self, seqs, attn, n_prompt, with_values=True):
        """Per-token log pi(o_t | s_t) and V(s_t) for every completion position t."""
        T = seqs.shape[1] - n_prompt
        out = self.lm(input_ids=seqs, attention_mask=attn, logits_to_keep=T + 1, output_hidden_states=with_values)
        logp = selective_log_softmax(out.logits[:, :-1], seqs[:, n_prompt:])
        values = None
        if with_values:
            h = out.hidden_states[-1][:, n_prompt - 1:-1]                         # state BEFORE each completion token
            values = self.value_head(h.float()).squeeze(-1)
        return logp, values

    @torch.no_grad()
    def ref_logprobs(self, seqs, attn, n_prompt):
        with self.lm.disable_adapter():
            return self.forward(seqs, attn, n_prompt, with_values=False)[0]


def gae(rewards, values, mask, lam=0.95, gamma=1.0):
    """Generalized Advantage Estimation over completion tokens; V after the last token is 0."""
    B, T = rewards.shape
    adv = torch.zeros_like(rewards); running = torch.zeros(B, device=rewards.device)
    for t in reversed(range(T)):
        next_v = values[:, t + 1] * mask[:, t + 1] if t + 1 < T else torch.zeros(B, device=rewards.device)
        cont = mask[:, t + 1] if t + 1 < T else torch.zeros(B, device=rewards.device)
        delta = rewards[:, t] + gamma * next_v - values[:, t]
        running = delta + gamma * lam * cont * running
        adv[:, t] = running * mask[:, t]
    return adv


def masked_mean(x, mask):
    return (x * mask).sum() / mask.sum().clamp(min=1)

def masked_whiten(x, mask):
    mu = masked_mean(x, mask); var = masked_mean((x - mu) ** 2, mask)
    return (x - mu) * torch.rsqrt(var + 1e-8) * mask


def ppo_minibatch_loss(model, seqs, attn, n_prompt, cmask, old_logp, old_values, adv, returns,
                       clip_eps=0.2, clip_v=0.2, vf_coef=0.1):
    logp, values = model(seqs, attn, n_prompt)
    ratio = torch.exp(logp - old_logp)
    pg_loss = masked_mean(torch.max(-adv * ratio, -adv * ratio.clamp(1 - clip_eps, 1 + clip_eps)), cmask)
    v_clipped = old_values + (values - old_values).clamp(-clip_v, clip_v)
    vf_loss = 0.5 * masked_mean(torch.max((values - returns) ** 2, (v_clipped - returns) ** 2), cmask)
    with torch.no_grad():
        stats = {"pg_loss": pg_loss.item(), "vf_loss": vf_loss.item(),
                 "clip_frac": masked_mean(((ratio - 1).abs() > clip_eps).float(), cmask).item(),
                 "approx_kl": masked_mean(torch.exp(old_logp - logp) - (old_logp - logp) - 1, cmask).item()}
    return pg_loss + vf_coef * vf_loss, stats

print("PPO components defined.")

### One PPO iteration, slowed down

The best way to understand PPO for LLMs is to print every intermediate tensor
for a single answer: the per-token rewards (almost all tiny negative KL
penalties, then one big number at the end), the critic's values, and the
per-token advantages GAE produces.

In [ ]:
set_seed(0)
ppo_model = PolicyValueModel(MODEL_NAME).to(DEVICE)
ppo_opt = torch.optim.AdamW([{"params": [p for p in ppo_model.lm.parameters() if p.requires_grad], "lr": 1e-5},
                             {"params": ppo_model.value_head.parameters(), "lr": 1e-4}])
print(f"trainable parameters: LoRA {count_trainable(ppo_model.lm) / 1e6:.2f}M + value head {count_trainable(ppo_model.value_head):,}")

P, BETA_KL, EPOCHS, MINI = 16, 0.04, 2, 4     # 16 prompts/iteration (same 16 samples/step as notebook 12)

@torch.no_grad()
def chunked(fn, seqs, attn, n_prompt, chunk=MINI):
    outs = [fn(seqs[i:i + chunk], attn[i:i + chunk], n_prompt) for i in range(0, len(seqs), chunk)]
    if isinstance(outs[0], tuple):
        return tuple(torch.cat(parts) for parts in zip(*outs))
    return torch.cat(outs)

def ppo_rollout(questions, answers):
    ppo_model.eval()
    seqs, attn, cmask, n_prompt, last, texts = ppo_model.sample(questions)
    score = torch.tensor([verifier_reward(t, a) for t, a in zip(texts, answers)], device=DEVICE)
    old_logp, old_values = chunked(ppo_model, seqs, attn, n_prompt)
    ref_logp = chunked(ppo_model.ref_logprobs, seqs, attn, n_prompt)
    kl_tok = (old_logp - ref_logp) * cmask                                    # k1, per token
    rewards = -BETA_KL * kl_tok
    rewards[torch.arange(len(seqs), device=DEVICE), last] += score            # sparse terminal reward
    old_values = old_values * cmask
    adv = gae(rewards, old_values, cmask)
    returns = adv + old_values
    return dict(seqs=seqs, attn=attn, cmask=cmask, n_prompt=n_prompt, texts=texts, score=score,
                old_logp=old_logp, old_values=old_values, adv=masked_whiten(adv, cmask), raw_adv=adv,
                returns=returns, kl_tok=kl_tok, rewards=rewards)

def ppo_update(b):
    ppo_model.train(); stats = defaultdict(list)
    for _ in range(EPOCHS):
        perm = torch.randperm(len(b["seqs"]), device=DEVICE)
        for i in range(0, len(perm), MINI):
            idx = perm[i:i + MINI]
            loss, s = ppo_minibatch_loss(ppo_model, b["seqs"][idx], b["attn"][idx], b["n_prompt"], b["cmask"][idx],
                                         b["old_logp"][idx], b["old_values"][idx], b["adv"][idx], b["returns"][idx])
            ppo_opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(ppo_model.parameters(), 1.0); ppo_opt.step()
            for k, v in s.items():
                stats[k].append(v)
    return {k: float(np.mean(v)) for k, v in stats.items()}

demo = random.Random(0).sample(TRAIN, P)
b = ppo_rollout([q for q, _ in demo], [a for _, a in demo])
i = int(torch.argmax(b["score"]))                                              # look at the best answer in the batch
n = int(b["cmask"][i].sum())
print(f"[1] rollout : {len(b['seqs'])} answers (one per prompt), lengths {b['cmask'].sum(1).int().tolist()}")
print(f"[2] scores  : {b['score'].cpu().numpy().round(2).tolist()}")
print(f"\n--- answer #{i} (score {b['score'][i]:.2f}, {n} tokens) ---\n{b['texts'][i][:300]}\n")
print("[3] per-token view of that answer (first 6 and last 4 tokens):")
print(f"{'t':>4} | {'token':>14} | {'reward r_t':>10} | {'value V(s_t)':>12} | {'advantage A_t':>13}")
toks = b["seqs"][i, b["n_prompt"]:]
for t in list(range(min(6, n))) + list(range(max(6, n - 4), n)):
    print(f"{t:>4} | {repr(tokenizer.decode(toks[t]))[:14]:>14} | {b['rewards'][i, t].item():+10.4f} | "
          f"{b['old_values'][i, t].item():+12.4f} | {b['raw_adv'][i, t].item():+13.4f}")
print("\n   -> rewards are ~0 (tiny KL penalties) until the LAST token, which carries the verifier score;")
print("      GAE propagates that score backwards so EARLIER tokens get credit too (V is ~0 before training).")
s = ppo_update(b)
print(f"\n[4] update  : {EPOCHS} epochs x {P // MINI} minibatches = {EPOCHS * P // MINI} optimizer steps on this batch")
print(f"              policy loss {s['pg_loss']:+.4f} | value loss {s['vf_loss']:.4f} | clip frac {s['clip_frac']:.3f} | approx KL {s['approx_kl']:.5f}")

Compare this with notebook 12's "one step, slowed down":

- **Per-token credit.** In GRPO every token of an answer shares one advantage.
  Here, each token's advantage $\hat{A}_t$ depends on the critic's estimate of
  how good the answer looked *at that point*. Once the critic has learned, a
  token where the answer "goes wrong" can get a negative advantage even
  inside an answer that ends well, and vice versa. That's the main thing PPO's
  critic buys.
- **Clip fraction > 0.** With 8 updates per batch the policy moves within
  the batch, so some ratios leave $[0.8, 1.2]$ and get clipped. In notebook 12
  (`MU = 1`) the clip fraction was always 0.

### The full PPO loop

In [ ]:
PPO_STEPS = 200          # x 16 prompts = 3,200 sampled answers: the same budget as notebook 12's GRPO
ppo_history = []
rng = random.Random(1)
t0 = time.time()
print(f"{'step':>4} | {'time':>6} | {'score':>6} | {'correct':>7} | {'format':>6} | {'length':>6} | {'KL/seq':>7} | "
      f"{'v-loss':>7} | {'clip':>5} | {'expl.var':>8}")
for step in range(PPO_STEPS):
    batch = rng.sample(TRAIN, P)
    b = ppo_rollout([q for q, _ in batch], [a for _, a in batch])
    s = ppo_update(b)
    with torch.no_grad():                                   # how well does the critic predict the returns?
        m = b["cmask"].bool()
        ev = 1 - torch.var(b["returns"][m] - b["old_values"][m]) / (torch.var(b["returns"][m]) + 1e-8)
    h = {"score": b["score"].mean().item(),
         "correct": float(np.mean([correctness_score(t, a) for t, (_, a) in zip(b["texts"], batch)])),
         "format": float(np.mean([format_score(t) for t in b["texts"]])),
         "length": b["cmask"].sum(1).mean().item(), "kl": b["kl_tok"].sum(1).mean().item(),
         "explained_var": ev.item(), **s}
    ppo_history.append(h)
    print(f"{step:>4} | {time.time() - t0:5.0f}s | {h['score']:6.3f} | {h['correct']:7.3f} | {h['format']:6.3f} | "
          f"{h['length']:6.1f} | {h['kl']:7.3f} | {h['vf_loss']:7.4f} | {h['clip_frac']:5.3f} | {h['explained_var']:8.3f}")
ppo_torch_minutes = (time.time() - t0) / 60
print(f"\nPPO (PyTorch) finished in {ppo_torch_minutes:.1f} min")

fig, axes = plt.subplots(1, 5, figsize=(24, 3.6))
for ax, (k, title) in zip(axes, [("correct", "correctness"), ("format", "format reward (max 0.75)"),
                                 ("length", "length (tokens)"), ("kl", "KL to reference (per answer)"),
                                 ("explained_var", "critic explained variance")]):
    ys = np.array([h[k] for h in ppo_history])
    ax.plot(ys, alpha=0.3, color="C2"); ax.plot(np.convolve(ys, np.ones(5) / 5, mode="valid"), color="C2")
    ax.set_title(title); ax.set_xlabel("step")
plt.suptitle("Part 1: PPO from scratch", y=1.03); plt.tight_layout(); plt.show()

ppo_torch_metrics, ppo_torch_responses = evaluate(ppo_model.lm, "AFTER PPO (PyTorch)")

**Reading the PPO-specific columns:**

- **KL/seq** is the *sum* over an answer's tokens, the quantity that is
  actually subtracted from the reward. It's not per token like notebook 12's log.
- **v-loss** is the critic's regression error. It falls as the critic learns to
  predict the final score from a partial answer.
- **expl.var** (explained variance) is $1 - \mathrm{Var}(R - V)/\mathrm{Var}(R)$.
  0 means the critic is no better than predicting a constant; values toward 1
  mean it predicts returns well. A critic stuck near 0 means GAE is giving
  little more than REINFORCE would.
- **clip** is the fraction of tokens whose ratio left $[0.8, 1.2]$ during the 8
  updates. If it's high (> 0.2–0.3), the updates are too large: lower the
  learning rate or the number of epochs.

In [ ]:
ppo_torch_trainable = count_trainable(ppo_model)
ppo_model.lm.save_pretrained(CKPT_DIR / "12_3_ppo_torch")
torch.save(ppo_model.value_head.state_dict(), CKPT_DIR / "12_3_ppo_torch" / "value_head.pt")
del ppo_opt
free_memory()

# Part 2 — PPO with `trl.experimental.ppo.PPOTrainer`

`trl`'s PPO is the classic RLHF design (notebooks 7 and 10). It holds:

| role | here |
|---|---|
| policy | Qwen + LoRA (passed via `peft_config`) |
| reference | the same model with adapters disabled (`ref_model=None` + LoRA) |
| **value model** | a **separate** `AutoModelForSequenceClassification` (full fine-tune) |
| **reward model** | expects a *model*: it calls `getattr(model, model.base_model_prefix)` for hidden states, then `model.score(...)`, and reads the score at the last response token |

### Wrapping the verifier as a "reward model"

We have a rule-based verifier, not a neural reward model. So we wrap it in an
`nn.Module` with the interface `trl` calls:

- `VerifierRewardModel.backbone(input_ids, attention_mask)` decodes each
  prompt + response, looks up the true answer for the question, computes notebook
  12's reward, and returns it as a fake "hidden state" of shape `(B, T, 1)`,
  with the same value at every position;
- `VerifierRewardModel.score` is `nn.Identity()`, so whichever position `trl`
  reads, it gets the verifier's reward.

This trick works for any programmatic reward (unit tests, regex checks, an
external judge API).

In [ ]:
class _VerifierBackbone(nn.Module):
    def forward(self, input_ids, attention_mask, **kwargs):
        rewards = []
        for ids, mask in zip(input_ids, attention_mask):
            text = tokenizer.decode(ids[mask.bool()], skip_special_tokens=False)
            question = text.split("<|im_start|>user\n")[-1].split("<|im_end|>")[0].strip()
            response = text.split("<|im_start|>assistant\n")[-1]
            response = response.replace("<|im_end|>", "").replace("<|endoftext|>", "").strip()
            rewards.append(verifier_reward(response, ANSWER_OF.get(question, "")))
        r = torch.tensor(rewards, dtype=torch.float32, device=input_ids.device)
        return type("Out", (), {"hidden_states": (r[:, None, None].expand(-1, input_ids.shape[1], 1),)})()

class VerifierRewardModel(nn.Module):
    """A rule-based reward with the interface trl's PPO expects from a reward model."""
    base_model_prefix = "backbone"

    def __init__(self):
        super().__init__()
        self.backbone = _VerifierBackbone()
        self.score = nn.Identity()

# sanity check: the wrapper must agree with the verifier
q, a = TRAIN[0]
probe = tokenizer([build_prompt(q) + f"<think>work</think><answer>{a}</answer><|im_end|>"], return_tensors="pt")
out = VerifierRewardModel().backbone(probe["input_ids"], probe["attention_mask"]).hidden_states[0]
print(f"wrapper reward: {out[0, -1, 0].item():.2f}  (expected {verifier_reward(f'<think>work</think><answer>{a}</answer>', a):.2f})")

In [ ]:
if HAS_TRL_PPO:
    def tokenize_prompts(problems):
        return Dataset.from_dict({"input_ids": [tokenizer(build_prompt(q), add_special_tokens=False)["input_ids"]
                                                for q, _ in problems]})

    ppo_train_ds, ppo_eval_ds = tokenize_prompts(TRAIN), tokenize_prompts(TEST)
    policy_trl = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE)
    value_trl = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1, dtype=torch.float32)
    value_trl.config.pad_token_id = tokenizer.pad_token_id

    ppo_config = PPOConfig(
        output_dir="/tmp/nb12_3_ppo_trl",
        per_device_train_batch_size=P, gradient_accumulation_steps=1,   # 16 prompts per rollout (as Part 1)
        num_mini_batches=P // MINI, num_ppo_epochs=EPOCHS,                # 4 minibatches x 2 epochs (as Part 1)
        total_episodes=PPO_STEPS * P,                                     # 3,200 sampled answers (as Part 1)
        response_length=160, stop_token="eos", temperature=1.0,
        kl_coef=BETA_KL, kl_estimator="k1",                               # per-token KL penalty in the reward
        cliprange=0.2, cliprange_value=0.2, vf_coef=0.1, gamma=1.0, lam=0.95,
        learning_rate=1e-5, local_rollout_forward_batch_size=P,
        num_sample_generations=0,                                         # we evaluate ourselves
        logging_steps=1, report_to="none", save_strategy="no", seed=0,
        bf16=USE_BF16, use_cpu=(DEVICE == "cpu"),
    )
    ppo_trainer = PPOTrainer(args=ppo_config, processing_class=tokenizer, model=policy_trl, ref_model=None,
                             reward_model=VerifierRewardModel(), value_model=value_trl,
                             train_dataset=ppo_train_ds, eval_dataset=ppo_eval_ds, peft_config=lora_cfg())
    print(f"policy trainable (LoRA): {count_trainable(ppo_trainer.policy_model) / 1e6:.2f}M | "
          f"value model trainable (full): {count_trainable(value_trl) / 1e6:.0f}M")
    t0 = time.time()
    ppo_trainer.train()
    ppo_trl_minutes = (time.time() - t0) / 60
    print(f"\nPPO (trl) finished in {ppo_trl_minutes:.1f} min")

    trl_ppo_log = [r for r in ppo_trainer.state.log_history if "objective/scores" in r]
    keys = {"objective/scores": "score", "objective/kl": "KL/seq", "loss/value_avg": "v-loss",
            "policy/clipfrac_avg": "clip", "policy/approxkl_avg": "approx KL", "policy/entropy_avg": "entropy"}
    print(f"\n{'#':>4} | " + " | ".join(f"{v:>10}" for v in keys.values()))
    for j, r in enumerate(trl_ppo_log):
        if j % 10 == 0 or j == len(trl_ppo_log) - 1:
            print(f"{j:>4} | " + " | ".join(f"{r.get(k, float('nan')):10.4f}" for k in keys))

    ppo_trl_policy = ppo_trainer.policy_model.eval()
    ppo_trl_metrics, ppo_trl_responses = evaluate(ppo_trl_policy, "AFTER PPO (trl)")
else:
    print("skipping trl PPO (trl.experimental.ppo unavailable)")

### Mapping `trl`'s PPO log to Part 1

| `trl` key | Part 1 column | meaning |
|---|---|---|
| `objective/scores` | score | mean verifier reward of the sampled answers |
| `objective/kl` | KL/seq | k1 KL to the reference, summed over each answer |
| `objective/non_score_reward` | — | $-\beta \cdot$ KL, the penalty part of the reward |
| `loss/value_avg` | v-loss | critic regression loss |
| `policy/clipfrac_avg` | clip | fraction of clipped ratios |
| `policy/approxkl_avg` | approx KL | how far each update moved the policy (old → new) |

In [ ]:
ppo_methods = {
    "PPO (PyTorch)": method_record(
        ppo_torch_metrics, ppo_torch_responses, algorithm="PPO (critic + GAE)", implementation="PyTorch",
        reward_signal="verifier (format + correctness)", train_minutes=ppo_torch_minutes,
        trainable_params=ppo_torch_trainable, samples_generated=PPO_STEPS * P,
        models_in_memory="policy (LoRA) + value head on the shared backbone; reference = adapters off"),
}
if HAS_TRL_PPO:
    ppo_methods["PPO (trl)"] = method_record(
        ppo_trl_metrics, ppo_trl_responses, algorithm="PPO (critic + GAE)", implementation="trl",
        reward_signal="verifier (format + correctness)", train_minutes=ppo_trl_minutes,
        trainable_params=count_trainable(ppo_trl_policy) + count_trainable(value_trl), samples_generated=PPO_STEPS * P,
        models_in_memory="policy (LoRA) + separate full value model; reference = adapters off")
    ppo_trl_policy.save_pretrained(CKPT_DIR / "12_3_ppo_trl")
ppo_methods["Base model (no training)"] = method_record(
    before_metrics, before_responses, algorithm="none", implementation="-", reward_signal="-",
    train_minutes=0, trainable_params=0, samples_generated=0, models_in_memory="-")
save_results("12_3_ppo.json", ppo_methods)

# Part 3 — The final comparison: GRPO vs RLCD vs DPO vs PPO

We load every saved result:

- `results/12_grpo.json` — notebook 12 (base, GRPO with `trl`, GRPO in PyTorch);
- `results/12_2_rlcd.json` — notebook 12.2 (RLCD + GRPO, RLCD + DPO with `trl`, RLCD + DPO in PyTorch);
- `results/12_3_ppo.json` — this notebook (PPO with PyTorch and `trl`).

Each file carries the **test-set fingerprint**. Results computed on a
different test set are refused. Any file that's missing (a notebook you
haven't run yet) is skipped.

In [ ]:
all_methods = {}
for fname in ["12_grpo.json", "12_2_rlcd.json", "12_3_ppo.json"]:
    path = RESULTS_DIR / fname
    if not path.exists():
        print(f"missing {path} -> run that notebook first (skipping)")
        continue
    payload = json.loads(path.read_text())
    if payload["test_fingerprint"] != TEST_FINGERPRINT:
        print(f"REFUSING {fname}: test fingerprint {payload['test_fingerprint']} != {TEST_FINGERPRINT}")
        continue
    for name, rec in payload["methods"].items():
        if name == "Base model (no training)" and name in all_methods:
            continue
        rec["source"] = fname
        all_methods[name] = rec
print(f"loaded {len(all_methods)} methods: {list(all_methods)}")

# Re-score every saved response with the SAME code, plus a "lenient tag" format check
# (notebook 12, FAQ 2: the model may reason in <thought> instead of <think>).
ANY_TAG_FORMAT_RE = re.compile(r"<(think|thought)>.+?</\1>\s*<answer>\s*-?\d+\s*</answer>", re.S)
answers = [a for _, a in TEST]
def rescore(responses):
    return {
        "strict acc": np.mean([parse_answer(r) == a for r, a in zip(responses, answers)]),
        "lenient acc": np.mean([re.findall(r"-?\d+", r)[-1:] == [a] for r, a in zip(responses, answers)]),
        "<answer> tag": np.mean([parse_answer(r) is not None for r in responses]),
        "format (<think>)": np.mean([follows_format(r) for r in responses]),
        "format (<think>|<thought>)": np.mean([bool(ANY_TAG_FORMAT_RE.search(r)) for r in responses]),
        "length (tok)": np.mean([len(tokenizer(r)["input_ids"]) for r in responses]),
    }
table = {name: rescore(rec["responses"]) for name, rec in all_methods.items()}
order = sorted(table, key=lambda n: -table[n]["strict acc"])
cols = list(next(iter(table.values())).keys())
print(f"\n{'method':28} | " + " | ".join(f"{c:>26}" for c in cols))
for n in order:
    print(f"{n:28} | " + " | ".join(f"{table[n][c]:26.3f}" if "length" not in c else f"{table[n][c]:26.1f}" for c in cols))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 4.5))
for ax, c in zip(axes, ["strict acc", "format (<think>|<thought>)", "length (tok)"]):
    vals = [table[n][c] for n in order]
    colors = ["gray" if "Base" in n else ("C0" if "GRPO" in n and "RLCD" not in n else
              ("C1" if "RLCD" in n else "C2")) for n in order]
    ax.barh(range(len(order)), vals, color=colors); ax.set_yticks(range(len(order))); ax.set_yticklabels(order, fontsize=8)
    ax.invert_yaxis(); ax.set_title(c)
plt.suptitle("gray = base | blue = GRPO + verifier | orange = RLCD (no verifier) | green = PPO + verifier", y=1.02)
plt.tight_layout(); plt.show()

### Cost and requirements

In [ ]:
print(f"{'method':28} | {'reward signal':42} | {'train min':>9} | {'trainable':>10} | {'samples':>8} | models in memory")
for n in order:
    r = all_methods[n]
    tm = "-" if r["train_minutes"] is None else f"{r['train_minutes']:.1f}"
    tp = "-" if r["trainable_params"] is None else f"{r['trainable_params'] / 1e6:.1f}M"
    sg = "-" if r["samples_generated"] is None else f"{r['samples_generated']:,}"
    print(f"{n:28} | {r['reward_signal'][:42]:42} | {tm:>9} | {tp:>10} | {sg:>8} | {r['models_in_memory']}")

### Which problems did each method fix, and do they fix the same ones?

In [ ]:
solved = {n: {i for i, (r, a) in enumerate(zip(all_methods[n]["responses"], answers)) if parse_answer(r) == a} for n in order}
base_key = "Base model (no training)"
base_solved = solved.get(base_key, set())
print(f"{'method':28} | {'solved':>6} | {'fixed vs base':>13} | {'broke vs base':>13}")
for n in order:
    print(f"{n:28} | {len(solved[n]):>6} | {len(solved[n] - base_solved):>13} | {len(base_solved - solved[n]):>13}")

trained = [n for n in order if n != base_key]
if len(trained) > 1:
    M = np.array([[len(solved[a] & solved[b]) / max(1, len(solved[a] | solved[b])) for b in trained] for a in trained])
    fig, ax = plt.subplots(figsize=(1.1 * len(trained) + 3, 1.0 * len(trained) + 2))
    im = ax.imshow(M, vmin=0, vmax=1, cmap="viridis")
    ax.set_xticks(range(len(trained))); ax.set_xticklabels(trained, rotation=60, ha="right", fontsize=8)
    ax.set_yticks(range(len(trained))); ax.set_yticklabels(trained, fontsize=8)
    for i in range(len(trained)):
        for j in range(len(trained)):
            ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", color="w", fontsize=7)
    ax.set_title("overlap of solved problems (Jaccard)"); plt.colorbar(im); plt.tight_layout(); plt.show()

never = [i for i in range(len(TEST)) if not any(i in solved[n] for n in trained)]
print(f"\nproblems no method solved: {len(never)}" + (f" -> e.g. {TEST[never[0]][0]} (answer {TEST[never[0]][1]})" if never else ""))

### The same question, every method

In [ ]:
for idx in [0, 2]:
    q, a = TEST[idx]
    print("=" * 110 + f"\nQUESTION: {q}\nTRUE ANSWER: {a}")
    for n in order:
        r = all_methods[n]["responses"][idx]
        verdict = "correct" if parse_answer(r) == a else ("wrong" if parse_answer(r) else "no answer")
        print(f"\n--- {n}  [{verdict}]\n{r.strip()[:350]}")

## How to read the comparison — and what each method teaches

| | GRPO + verifier (12) | RLCD + GRPO (12.2) | RLCD + DPO (12.2) | PPO + verifier (12.3) |
|---|---|---|---|---|
| **what defines "good"** | a program that checks the answer | a reward model learned from contrastive pairs | the contrastive pairs directly | a program that checks the answer |
| **needs a verifier?** | yes | **no** | **no** | yes |
| **needs a reward model?** | no | yes | no | no (rule-based here) |
| **needs a critic?** | no (group baseline) | no | no | **yes** (value head / model) |
| **credit assignment** | one advantage per answer | one advantage per answer | implicit, per pair | **per token** (GAE) |
| **on-policy sampling?** | yes | yes | **no** (offline pairs) | yes |
| **KL to reference** | k3 in the loss | k3 in the loss | implicit through $\beta$ | k1 in the per-token reward |

**Questions to ask of your results:**

1. **GRPO vs PPO (same reward, same budget).** If their accuracies are close,
   the critic didn't buy much. That's common when the reward is a single score
   at the end and answers are short: the group baseline already works well, and
   the critic needs many steps to become useful (watch PPO's explained
   variance). If PPO is clearly better, per-token credit assignment helped. If
   it's clearly worse, the critic was probably still inaccurate, adding bias
   (notebook 5).
2. **Verifier vs no verifier (GRPO vs RLCD).** How much of the GRPO gain does
   RLCD recover *without ever checking an answer*? Whatever it recovers is the
   part of "correct reasoning" captured by "careful vs careless" style. The
   rest needs a real correctness signal.
3. **RL vs DPO on the same RLCD pairs.** The RL route keeps sampling from the
   improving policy. DPO only sees the original pairs. Check whether RLCD+GRPO
   beats RLCD+DPO on accuracy, and whether it paid for that in KL drift or
   length.
4. **`trl` vs PyTorch for the same algorithm.** They should be close. Gaps of a
   few problems are within noise: 48 problems ≈ ±7 percentage points. Larger gaps point to
   real implementation differences (shared vs. separate value network,
   normalization, the order of examples).
5. **Format vs correctness.** Use the two format columns. If `<think>|<thought>`
   is high but `<think>` is low, the model learned to reason in its own
   preferred tags (notebook 12, FAQ 2). The reasoning is there; the strict
   checker just doesn't count it.

### Rules of thumb these notebooks support

- **Have a verifier? Use it.** GRPO (or RLOO) with a verifiable reward is the
  simplest strong baseline: no critic, no reward model, little to tune.
- **No verifier, but you can describe "better vs worse"?** RLCD gives you
  preference data for free. Check the contrast diagnostic, probe the reward
  model for shortcuts, and monitor a true metric if you have *any*.
- **PPO** earns its extra machinery when rewards are dense or responses are
  long enough that per-token credit matters, or when you already run a reward
  model. For short answers with a single terminal score, it's often matched by
  GRPO at lower cost.
- **DPO** is the cheapest route from pairs to a better policy, with the
  offline caveats from notebook 10.

## Exercises

1. **Separate critic.** Replace Part 1's shared value head with a separate
   `AutoModelForSequenceClassification` (as `trl` does). Does the explained
   variance rise faster? What does it cost in memory?
2. **Fewer PPO epochs.** Set `EPOCHS = 1, MINI = P` (one update per batch). The
   clip fraction should drop to 0. How does accuracy change?
3. **Train longer, compare curves.** Double every method's budget and plot
   strict accuracy against samples generated, on one figure.
4. **Transfer.** Evaluate all saved adapters (in `checkpoints/`) on 50 GSM8K
   questions. Which method's reasoning transfers best beyond the training
   templates?